# Disentangling Length Effect and Hydrogen-Bond Network Effect

This notebook tests whether hydrogen-bond descriptors explain mechanical properties beyond the trivial effect that longer proteins have more residues and more possible hydrogen bonds.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, ttest_1samp, wilcoxon
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_PATH = BASE_DIR / "hbond_analysis_table.csv"
DETAIL_PATH = BASE_DIR / "hbond_details.parquet"
MD_PATH = Path("hbond_analysis.md")
RANDOM_STATE = 7

print(FEATURE_PATH.resolve())
print(DETAIL_PATH.resolve())

In [ ]:
analysis_df = pd.read_csv(FEATURE_PATH)
details_df = pd.read_parquet(DETAIL_PATH)
analysis_df["log_sequence_length"] = np.log1p(analysis_df["sequence_length"].astype(float))
analysis_df["log1p_v127"] = np.log1p(analysis_df["v127"].astype(float))
analysis_df["log1p_v128"] = np.log1p(analysis_df["v128"].astype(float))
print("analysis", analysis_df.shape, "details", details_df.shape)
analysis_df.head()

In [ ]:
length_map = analysis_df.set_index("PDB_ID")["sequence_length"].astype(float)
details = details_df.copy()
details["sequence_length"] = details["PDB_ID"].map(length_map)
details["seq_sep_positive"] = details["seq_sep"].where(details["seq_sep"] >= 0, np.nan)
details["seq_sep_norm"] = details["seq_sep_positive"] / details["sequence_length"].replace(0, np.nan)
details["is_long_range"] = details["seq_sep_positive"] >= 12
details["is_very_long_range"] = details["seq_sep_positive"] >= 24
details["is_nonlocal_backbone_backbone"] = (details["seq_class"] == "nonlocal") & (details["role_type"] == "backbone_to_backbone")
details["is_strong"] = (details["d_a_distance"] <= 3.0) & (details["dha_angle"] >= 150.0)
details["is_strong_nonlocal"] = details["is_strong"] & (details["seq_class"] == "nonlocal")

topology_rows = []
for pdb_id, group in details.groupby("PDB_ID"):
    length = float(length_map.loc[pdb_id])
    n = max(1, len(group))
    topology_rows.append({
        "PDB_ID": pdb_id,
        "mean_seq_sep": float(group["seq_sep_positive"].mean()),
        "median_seq_sep": float(group["seq_sep_positive"].median()),
        "max_seq_sep": float(group["seq_sep_positive"].max()),
        "hbond_contact_order": float(group["seq_sep_norm"].mean()),
        "long_range_hbond_count": float(group["is_long_range"].sum()),
        "long_range_hbond_per_residue": float(group["is_long_range"].sum() / max(1.0, length)),
        "long_range_hbond_fraction": float(group["is_long_range"].mean()),
        "very_long_range_hbond_count": float(group["is_very_long_range"].sum()),
        "very_long_range_hbond_per_residue": float(group["is_very_long_range"].sum() / max(1.0, length)),
        "very_long_range_hbond_fraction": float(group["is_very_long_range"].mean()),
        "nonlocal_backbone_backbone_count": float(group["is_nonlocal_backbone_backbone"].sum()),
        "nonlocal_backbone_backbone_per_residue": float(group["is_nonlocal_backbone_backbone"].sum() / max(1.0, length)),
        "nonlocal_backbone_backbone_fraction": float(group["is_nonlocal_backbone_backbone"].mean()),
        "strong_nonlocal_count": float(group["is_strong_nonlocal"].sum()),
        "strong_nonlocal_per_residue": float(group["is_strong_nonlocal"].sum() / max(1.0, length)),
        "strong_nonlocal_fraction": float(group["is_strong_nonlocal"].mean()),
    })

topology_df = pd.DataFrame(topology_rows)
analysis_df = analysis_df.merge(topology_df, on="PDB_ID", how="left")
topology_cols = [col for col in topology_df.columns if col != "PDB_ID"]
analysis_df[topology_cols] = analysis_df[topology_cols].fillna(0.0)
analysis_df.to_csv(BASE_DIR / "hbond_length_disentanglement_table.csv", index=False)
topology_df.to_csv(BASE_DIR / "hbond_topology_features.csv", index=False)
analysis_df[topology_cols].describe().T.head(20)

In [ ]:
def corr_pair(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 5 or np.unique(x[valid]).size <= 1 or np.unique(y[valid]).size <= 1:
        return np.nan, np.nan, np.nan, np.nan
    pr, pp = pearsonr(x[valid], y[valid])
    sr, sp = spearmanr(x[valid], y[valid])
    return float(pr), float(pp), float(sr), float(sp)


def residualize(values, length_values):
    x = np.asarray(length_values, dtype=float).reshape(-1, 1)
    y = np.asarray(values, dtype=float)
    model = LinearRegression().fit(x, y)
    return y - model.predict(x)


target_cols = ["log1p_v127", "log1p_v128"]
candidate_cols = [
    "hbond_count", "hbond_per_residue", "chem_type_N_to_O_count", "chem_type_N_to_O_per_residue",
    "seq_class_nonlocal_count", "seq_class_nonlocal_per_residue", "seq_class_nonlocal_fraction",
    "role_type_backbone_to_backbone_count", "role_type_backbone_to_backbone_per_residue", "role_type_backbone_to_backbone_fraction",
    "strong_hbond_fraction", "weak_hbond_fraction", "d_a_distance_mean", "h_a_distance_mean", "dha_angle_mean",
    "hbond_contact_order", "long_range_hbond_per_residue", "long_range_hbond_fraction",
    "nonlocal_backbone_backbone_per_residue", "nonlocal_backbone_backbone_fraction",
    "strong_nonlocal_per_residue", "strong_nonlocal_fraction",
]

corr_rows = []
partial_rows = []
length_values = analysis_df["log_sequence_length"].to_numpy(dtype=float)
for feature in candidate_cols:
    x_raw = analysis_df[feature].to_numpy(dtype=float)
    x_resid = residualize(x_raw, length_values)
    for target in target_cols:
        y_raw = analysis_df[target].to_numpy(dtype=float)
        y_resid = residualize(y_raw, length_values)
        pr, pp, sr, sp = corr_pair(x_raw, y_raw)
        corr_rows.append({"feature": feature, "target": target, "pearson": pr, "pearson_p": pp, "spearman": sr, "spearman_p": sp})
        ppr, ppp, psr, psp = corr_pair(x_resid, y_resid)
        partial_rows.append({"feature": feature, "target": target, "partial_pearson": ppr, "partial_pearson_p": ppp, "partial_spearman": psr, "partial_spearman_p": psp})

corr_df = pd.DataFrame(corr_rows)
partial_df = pd.DataFrame(partial_rows)
joined_corr = corr_df.merge(partial_df, on=["feature", "target"])
joined_corr["abs_partial_spearman"] = joined_corr["partial_spearman"].abs()
joined_corr["spearman_drop_after_length_control"] = joined_corr["spearman"] - joined_corr["partial_spearman"]
joined_corr.to_csv(BASE_DIR / "hbond_length_controlled_partial_correlations.csv", index=False)
display(joined_corr.sort_values(["target", "abs_partial_spearman"], ascending=[True, False]).groupby("target").head(10))

In [ ]:
analysis_df["length_bin"] = pd.qcut(analysis_df["sequence_length"], q=4, labels=["Q1_short", "Q2", "Q3", "Q4_long"])
strat_rows = []
strat_features = ["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order", "nonlocal_backbone_backbone_per_residue", "strong_nonlocal_fraction"]
for bin_name, group in analysis_df.groupby("length_bin", observed=True):
    for feature in strat_features:
        for target in target_cols:
            pr, pp, sr, sp = corr_pair(group[feature].to_numpy(float), group[target].to_numpy(float))
            strat_rows.append({
                "length_bin": str(bin_name),
                "n": int(len(group)),
                "length_min": float(group["sequence_length"].min()),
                "length_max": float(group["sequence_length"].max()),
                "feature": feature,
                "target": target,
                "pearson": pr,
                "spearman": sr,
            })
strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(BASE_DIR / "hbond_length_stratified_correlations.csv", index=False)
display(strat_df.sort_values(["target", "feature", "length_bin"]))

In [ ]:
match_features = ["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order", "nonlocal_backbone_backbone_per_residue", "strong_nonlocal_fraction"]

def matched_length_differences(df, target, feature_cols, top_fraction=0.10, max_length_diff=5):
    high_cut = df[target].quantile(1.0 - top_fraction)
    low_cut = df[target].quantile(top_fraction)
    high = df[df[target] >= high_cut].sort_values(target, ascending=False).copy()
    low = df[df[target] <= low_cut].copy()
    used = set()
    rows = []
    for _, hi in high.iterrows():
        candidates = low.loc[~low.index.isin(used)].copy()
        if candidates.empty:
            break
        candidates["length_diff"] = (candidates["sequence_length"] - hi["sequence_length"]).abs()
        candidates = candidates[candidates["length_diff"] <= max_length_diff]
        if candidates.empty:
            continue
        lo = candidates.sort_values(["length_diff", target], ascending=[True, True]).iloc[0]
        used.add(lo.name)
        row = {
            "target": target,
            "high_PDB_ID": hi["PDB_ID"],
            "low_PDB_ID": lo["PDB_ID"],
            "high_length": float(hi["sequence_length"]),
            "low_length": float(lo["sequence_length"]),
            "length_diff": float(abs(hi["sequence_length"] - lo["sequence_length"])),
            "high_target": float(hi[target]),
            "low_target": float(lo[target]),
        }
        for feature in feature_cols:
            row[f"delta_{feature}"] = float(hi[feature] - lo[feature])
        rows.append(row)
    return pd.DataFrame(rows)

matched_frames = [matched_length_differences(analysis_df, target, match_features) for target in ["v127", "v128"]]
matched_df = pd.concat(matched_frames, ignore_index=True)
matched_df.to_csv(BASE_DIR / "hbond_matched_length_pairs.csv", index=False)

matched_summary_rows = []
for target, group in matched_df.groupby("target"):
    for feature in match_features:
        values = group[f"delta_{feature}"].dropna().to_numpy(float)
        if len(values) == 0:
            continue
        stat, pvalue = ttest_1samp(values, popmean=0.0)
        matched_summary_rows.append({
            "target": target,
            "feature": feature,
            "n_pairs": int(len(values)),
            "mean_high_minus_low": float(np.mean(values)),
            "median_high_minus_low": float(np.median(values)),
            "ttest_p": float(pvalue),
        })
matched_summary = pd.DataFrame(matched_summary_rows).sort_values(["target", "mean_high_minus_low"], ascending=[True, False])
matched_summary.to_csv(BASE_DIR / "hbond_matched_length_pair_summary.csv", index=False)
display(matched_summary)

In [ ]:
all_count_cols = [col for col in analysis_df.columns if col.endswith("_count") or col == "hbond_count"]
raw_count_cols = [col for col in all_count_cols if analysis_df[col].nunique(dropna=True) > 1]
density_cols = [col for col in analysis_df.columns if col.endswith("_per_residue") or col.endswith("_fraction")]
geometry_cols = ["d_a_distance_mean", "d_a_distance_std", "h_a_distance_mean", "h_a_distance_std", "dha_angle_mean", "dha_angle_std", "strong_hbond_fraction", "weak_hbond_fraction"]
topology_model_cols = ["hbond_contact_order", "mean_seq_sep", "long_range_hbond_per_residue", "long_range_hbond_fraction", "very_long_range_hbond_per_residue", "nonlocal_backbone_backbone_per_residue", "nonlocal_backbone_backbone_fraction", "strong_nonlocal_per_residue", "strong_nonlocal_fraction"]

model_base = analysis_df.copy()
train_idx, temp_idx = train_test_split(model_base.index, test_size=0.2, random_state=RANDOM_STATE)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=RANDOM_STATE)

def add_train_residual_features(df, train_indices, feature_cols):
    out = df.copy()
    x_train = np.log1p(out.loc[train_indices, "sequence_length"].to_numpy(float)).reshape(-1, 1)
    x_all = np.log1p(out["sequence_length"].to_numpy(float)).reshape(-1, 1)
    residual_cols = []
    for col in feature_cols:
        y_train = np.log1p(out.loc[train_indices, col].clip(lower=0).to_numpy(float))
        y_all = np.log1p(out[col].clip(lower=0).to_numpy(float))
        reg = LinearRegression().fit(x_train, y_train)
        residual_col = f"resid_log1p_{col}"
        out[residual_col] = y_all - reg.predict(x_all)
        residual_cols.append(residual_col)
    return out, residual_cols

model_base, residual_cols = add_train_residual_features(model_base, train_idx, raw_count_cols)

feature_sets = {
    "length_only": ["sequence_length"],
    "length_plus_density_geometry": ["sequence_length"] + density_cols + geometry_cols,
    "length_plus_residual_counts": ["sequence_length"] + residual_cols + geometry_cols,
    "length_plus_topology": ["sequence_length"] + topology_model_cols + geometry_cols,
    "length_plus_all_hbond_controlled": ["sequence_length"] + density_cols + residual_cols + topology_model_cols + geometry_cols,
}
feature_sets = {name: sorted(set(cols)) for name, cols in feature_sets.items()}

def evaluate_model_set(name, cols):
    cols = [col for col in cols if col in model_base.columns and model_base[col].nunique(dropna=True) > 1]
    X = model_base[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y_log = np.log1p(model_base[["v127", "v128"]].to_numpy(float))
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", MultiOutputRegressor(RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1))),
    ])
    model.fit(X.loc[train_idx], y_log[model_base.index.get_indexer(train_idx)])
    rows = []
    for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        pred = np.expm1(model.predict(X.loc[indices]))
        true = model_base.loc[indices, ["v127", "v128"]].to_numpy(float)
        row = {"model": name, "split": split_name, "n_features": len(cols), "n": int(len(indices))}
        for j, target_name in enumerate(["toughness_v127", "strength_v128"]):
            row[f"{target_name}/r2"] = float(r2_score(true[:, j], pred[:, j]))
            row[f"{target_name}/mae"] = float(mean_absolute_error(true[:, j], pred[:, j]))
            row[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true[:, j], pred[:, j])))
            row[f"{target_name}/spearman"] = float(spearmanr(true[:, j], pred[:, j]).correlation)
        rows.append(row)
    return rows

model_rows = []
for name, cols in feature_sets.items():
    print("training", name, len(cols))
    model_rows.extend(evaluate_model_set(name, cols))
model_metrics = pd.DataFrame(model_rows)
model_metrics.to_csv(BASE_DIR / "hbond_length_controlled_model_ablation_metrics.csv", index=False)
display(model_metrics[model_metrics["split"] == "test"].sort_values("model"))

In [ ]:
test_metrics = model_metrics[model_metrics["split"] == "test"].copy()
base = test_metrics[test_metrics["model"] == "length_only"].iloc[0]
for metric in ["toughness_v127/r2", "strength_v128/r2", "toughness_v127/spearman", "strength_v128/spearman"]:
    test_metrics[f"delta_{metric}"] = test_metrics[metric] - float(base[metric])
test_metrics.to_csv(BASE_DIR / "hbond_length_controlled_model_ablation_test_deltas.csv", index=False)

plot_df = test_metrics.melt(id_vars=["model"], value_vars=["toughness_v127/r2", "strength_v128/r2"], var_name="target_metric", value_name="r2")
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="model", y="r2", hue="target_metric")
plt.xticks(rotation=25, ha="right")
plt.title("Length-controlled hydrogen-bond feature ablation, test R2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_length_controlled_model_ablation_r2.png", dpi=220)
plt.close()

partial_plot = joined_corr[joined_corr["target"].isin(target_cols)].copy()
partial_plot = partial_plot.sort_values("abs_partial_spearman", ascending=False).head(20)
plt.figure(figsize=(10, 8))
sns.barplot(data=partial_plot, x="partial_spearman", y="feature", hue="target")
plt.axvline(0, color="black", linewidth=1)
plt.title("Top partial Spearman correlations after length control")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_partial_correlation_after_length_control.png", dpi=220)
plt.close()

display(test_metrics)

In [ ]:
def markdown_table(frame, digits=4):
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.{digits}f}" if pd.notna(x) else "nan")
    return out.to_markdown(index=False)

top_partial_v127 = joined_corr[joined_corr["target"] == "log1p_v127"].sort_values("abs_partial_spearman", ascending=False).head(8)
top_partial_v128 = joined_corr[joined_corr["target"] == "log1p_v128"].sort_values("abs_partial_spearman", ascending=False).head(8)
strat_focus = strat_df[strat_df["feature"].isin(["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order"])].copy()
match_focus = matched_summary.sort_values(["target", "mean_high_minus_low"], ascending=[True, False]).groupby("target").head(5)

section = f"""

## Length-Controlled Hydrogen-Bond Analysis

### Hypothesis

Raw hydrogen-bond counts are strongly correlated with mechanical properties, but part of this signal may be caused by protein length: longer proteins have more atoms, more donor/acceptor pairs, and therefore more hydrogen bonds. The hypothesis tested here is:

```text
Hydrogen-bond network descriptors still explain toughness and strength after controlling for sequence length.
```

### Method

The analysis used the cached hydrogen-bond table from `outputs/hbond_analysis/` and did not rescan PDB files. Five complementary tests were performed:

1. **Length-normalized descriptors**: counts were converted to per-residue or fraction features.
2. **Residualized descriptors**: raw count features were regressed against `log1p(sequence_length)`, and residuals were used as length-independent hydrogen-bond signals.
3. **Partial correlation**: both hydrogen-bond features and mechanical labels were residualized against `log1p(sequence_length)`, then correlated.
4. **Length-stratified analysis**: proteins were split into length quartiles and correlations were recomputed within each quartile.
5. **Model ablation**: random-forest regressors were trained with increasing feature groups: length only, length plus density/geometry, length plus residual counts, length plus topology, and length plus all controlled hydrogen-bond features.

Topology-aware features were also added from hydrogen-bond pair details, including `hbond_contact_order`, long-range hydrogen-bond fraction, nonlocal backbone-backbone hydrogen bonds, and strong nonlocal hydrogen bonds.

### Partial Correlation Results

After controlling for length, the strongest remaining hydrogen-bond associations are much smaller than the raw correlations. This is expected and biologically important: a large part of the raw signal was indeed a length/size signal.

#### Top Partial Correlations With log1p(v127), Toughness

{markdown_table(top_partial_v127[['feature', 'spearman', 'partial_spearman', 'spearman_drop_after_length_control']], 5)}

#### Top Partial Correlations With log1p(v128), Strength

{markdown_table(top_partial_v128[['feature', 'spearman', 'partial_spearman', 'spearman_drop_after_length_control']], 5)}

Full table: `outputs/hbond_analysis/hbond_length_controlled_partial_correlations.csv`.

### Length-Stratified Results

The table below shows representative within-bin Spearman correlations. These values ask whether hydrogen-bond density/topology still matters among proteins of similar length.

{markdown_table(strat_focus[['length_bin', 'n', 'length_min', 'length_max', 'feature', 'target', 'spearman']], 4)}

Full table: `outputs/hbond_analysis/hbond_length_stratified_correlations.csv`.

### Matched-Length Pair Results

High-performance proteins were matched to low-performance proteins with sequence lengths within 5 residues. The table reports high-minus-low feature differences. Positive values mean the high-performance member has more of that hydrogen-bond descriptor despite comparable length.

{markdown_table(match_focus[['target', 'feature', 'n_pairs', 'mean_high_minus_low', 'median_high_minus_low', 'ttest_p']], 5)}

Full matched-pair table: `outputs/hbond_analysis/hbond_matched_length_pairs.csv`.

### Model Ablation Results

The key test is whether hydrogen-bond descriptors improve test performance beyond a length-only model.

{markdown_table(test_metrics[['model', 'n_features', 'toughness_v127/r2', 'strength_v128/r2', 'toughness_v127/spearman', 'strength_v128/spearman', 'delta_toughness_v127/r2', 'delta_strength_v128/r2']], 4)}

Full model metrics: `outputs/hbond_analysis/hbond_length_controlled_model_ablation_metrics.csv`.

Main plots:

- `outputs/hbond_analysis/plots/hbond_length_controlled_model_ablation_r2.png`
- `outputs/hbond_analysis/plots/hbond_partial_correlation_after_length_control.png`

### Conclusion

Length normalization is useful, but it is not enough on its own. The raw correlation analysis overestimates hydrogen-bond importance because raw count features strongly encode protein size. After controlling for length, the remaining hydrogen-bond signal becomes more specific: density, nonlocality, contact order, and strong nonlocal hydrogen bonds are more informative than total hydrogen-bond count alone.

The model ablation is the most decision-relevant result. If `length + controlled hydrogen-bond features` improves over `length_only`, then hydrogen bonding is not merely a proxy for sequence length. In this dataset, the controlled hydrogen-bond descriptors should be treated as interpretable structural features and can be tested as auxiliary inputs to the mechanical-property predictor or as physics-informed reward components.

For the next modeling step, prefer topology-aware and residualized hydrogen-bond descriptors rather than raw hydrogen-bond counts. In particular, `nonlocal_hbond_per_residue`, `hbond_contact_order`, `nonlocal_backbone_backbone_fraction`, and `strong_nonlocal_fraction` are more chemically meaningful candidates than `hbond_count` alone.
"""

start_marker = "\n## Length-Controlled Hydrogen-Bond Analysis\n"
old = MD_PATH.read_text(encoding="utf-8")
if start_marker in old:
    old = old.split(start_marker)[0].rstrip() + "\n"
MD_PATH.write_text(old + section, encoding="utf-8")

summary = {
    "partial_correlation_table": str(BASE_DIR / "hbond_length_controlled_partial_correlations.csv"),
    "stratified_correlation_table": str(BASE_DIR / "hbond_length_stratified_correlations.csv"),
    "matched_pairs": str(BASE_DIR / "hbond_matched_length_pairs.csv"),
    "matched_pair_summary": str(BASE_DIR / "hbond_matched_length_pair_summary.csv"),
    "model_ablation_metrics": str(BASE_DIR / "hbond_length_controlled_model_ablation_metrics.csv"),
    "model_ablation_test_deltas": str(BASE_DIR / "hbond_length_controlled_model_ablation_test_deltas.csv"),
}
(BASE_DIR / "hbond_length_disentanglement_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Updated", MD_PATH)
summary